<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks/4.2-feature-extraction-rav_and_crema.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Alo alo vecinillo

In [6]:
import kagglehub
import os
from google.colab import drive
drive.mount('/content/drive')
# Download latest version desde Kaggle
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
path_crema = kagglehub.dataset_download("ejlok1/cremad")
FAST_ROOT_DIR = '/content/features_local/ravdess-and-crema-images'
BASE_DIR_RAV = "/kaggle/input/ravdess-emotional-speech-audio"
BASE_DIR_CREMA = "/kaggle/input/cremad"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Using Colab cache for faster access to the 'cremad' dataset.


In [2]:
# Import de librerias necesarias

import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import logging
import sys

In [11]:
# Copiamos la carpeta entera de features desde Drive al disco de Colab
#!cp -r /content/drive/MyDrive/ravdess_images_02/ /content/features_local
#os.makedirs(FAST_ROOT_DIR, exist_ok=True)
# Opcional: Si tienes un archivo .zip en Drive, es AÚN MÁS RÁPIDO copiar el .zip y descomprimirlo localmente:
!cp -r /kaggle/input/ravdess-emotional-speech-audio /content/features_local/ravdess-and-crema-images
!cp -r /kaggle/input/cremad /content/features_local/ravdess-and-crema-images
#!unzip -q /content/ravdess_images_02.zip -d /content/features_local


In [12]:
# Unificando convecion de nombres y archivos antes de procesar los clips
os.listdir(FAST_ROOT_DIR)

['ravdess-emotional-speech-audio', 'cremad']

In [13]:
# Extraccion de caracteristicas (De audio a: dataframe(csv) - imagen(png)):

# Variables de configuración
#--------------------------------------------------------------------------
SAMPLE_RATE = 22050
MIN_DURATION = 0.5
MAX_DURATION = 3
GENERATE_CSV = False
GENERATE_IMAGES = False # Poner en False para solo obtener DF (Parte CSV)
PAD_MODE = "constant" # Para padding de audios cortos
OUT_DIR_IMAGES = '/content/drive/MyDrive/ravdess_and_crema_images/'
OUT_DIR_CSV = '/content/drive/MyDrive/ravdess_features/features_extended.csv'
BATCH_SIZE_CSV = 60 # Depende cuanto RAM usemos
IMG_RES = (224,224) # Para ResNet
file_path = FAST_ROOT_DIR

#--------------------------------------------------------------------------

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    stream=sys.stdout,
                    force=True
                    )

#Contadores para proceso de los clips

processed_count=0
discarded_count=0
error_count=0

# Funciones para la extraccion:

# Sabemos que en eg. RAVDESS: 03-01-XX-01-01-01-01.wav -> XX= corresponde a la emoción

# Función para agrupar por emoción

def get_emotion_from_filename(filename):
    try:
        parts = filename.split('-')
        emotion_code = int(parts[2]) # Posicion donde está XX, se ha descartado calm
        emotion_map = {1: 'neutral', 3: 'happy', 4: 'sad', 5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised'}
        return emotion_map.get(emotion_code, 'unknown')
    except:
        return 'unknown'

# Función process_audio
def process_audio(file_path, sr=SAMPLE_RATE, min_dur=MIN_DURATION, max_dur=MAX_DURATION):
    global discarded_count, error_count
    try:
        # Como vimos, es mas conveninete no ser tan agresivos y se procede a usar 40 en top_db
        y, sr = librosa.load(file_path, sr=sr)
        y_trimmed, _ = librosa.effects.trim(y, top_db=45)
        y_trimmed = librosa.util.normalize(y_trimmed)
        duration = len(y_trimmed) / sr
        target_len = int(max_dur * sr)
        y = y_trimmed


        if duration < min_dur:
            logging.info(f"Descartado por insignificante xd {os.path.basename(file_path)}: duración {duration:.2f}s")
            discarded_count += 1
            return None


        #  Escencial en operaciones matriciales

        if len(y_trimmed) < target_len:
            y_trimmed = np.pad(y_trimmed, (0, target_len - len(y_trimmed)), mode=PAD_MODE)

        return y_trimmed, sr
    except Exception as e:
        logging.error(f"Error en {file_path}: {e}")
        discarded_count += 1
        return None


def extract_features(y, sr):
    try:
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20) # Vamos esta vez con 30 coeficientes de Mel (filtros)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048)
        mel_spec_db = librosa.power_to_db(mel_spec)
        rmse = librosa.feature.rms(y=y)      # Solo para CSV
        delta = librosa.feature.delta(mfccs) # Solo para CSV
        delta2 = librosa.feature.delta(mfccs, order=2)  # Solo para CSV

        features = {
            'mfcc_mean': np.mean(mfccs, axis=1), 'mfcc_std': np.std(mfccs, axis=1),
            'delta_mean': np.mean(delta, axis=1), 'delta_std': np.std(delta, axis=1),
            'delta2_mean': np.mean(delta2, axis=1), 'delta2_std': np.std(delta2, axis=1),
            'rmse_mean': np.mean(rmse), 'rmse_std': np.std(rmse),
            'spec_mean': np.mean(spec_db), 'spec_std': np.std(spec_db),
            'mel_mean': np.mean(mel_spec_db), 'mel_std': np.std(mel_spec_db)
        }
        flat_features = np.concatenate([
            features['mfcc_mean'], features['mfcc_std'], features['delta_mean'], features['delta_std'],
            features['delta2_mean'], features['delta2_std'],
            [features['rmse_mean'], features['rmse_std'], features['spec_mean'], features['spec_std'],
             features['mel_mean'], features['mel_std']]
        ])
        return flat_features, mfccs, delta, delta2, spec_db, mel_spec_db
    except Exception as e:
        logging.error(f"Error features: {e}")
        return None, None, None, None, None, None


# Debug inicial
print(f"Path del dataset: {DATASET_PATH}")
print(f"Duración rango: {MIN_DURATION}s - {MAX_DURATION}s")
print(f"Generar imágenes: {GENERATE_IMAGES}")


# Main con lotes
data_dir = DATASET_PATH
files_list = [os.path.join(root, file) for root, dirs, files in os.walk(data_dir) for file in files if file.endswith('.wav')]
logging.info(f"Total archivos: {len(files_list)}")

#Bandera para no genrar nuevamente el csv:
if GENERATE_CSV:


    features_list = []
    labels = []
    for i in range(0, len(files_list), BATCH_SIZE_CSV):
        batch_files = files_list[i:i+BATCH_SIZE_CSV]
        logging.info(f"Procesando batch {i//BATCH_SIZE_CSV + 1} / {(len(files_list)//BATCH_SIZE_CSV) + 1}")
        for file_path in batch_files:
            processed = process_audio(file_path)
            if processed:
                y, sr = processed
                feats, mfccs, delta, delta2, spec_db, mel_spec_db = extract_features(y, sr)
                if feats is not None:
                    features_list.append(feats)
                    emotion = get_emotion_from_filename(os.path.basename(file_path))
                    labels.append(emotion)
                    processed_count += 1


        # Liberar memoria
        import gc
        gc.collect()
        logging.info(f"Batch {i//BATCH_SIZE_CSV + 1} completado. Procesados acumulados: {processed_count}, Descartados: {discarded_count}, Errores: {error_count}")

    # Estadísticas finales
    logging.info(f"Procesamiento finalizado. Total procesados: {processed_count}, Descartados: {discarded_count}, Errores: {error_count}")
    print(f"Resumen: Procesados {processed_count}, Descartados {discarded_count}, Errores {error_count}")


    # Crear y guardar DF
    columns = ([f'mfcc_mean_{i}' for i in range(13)] + [f'mfcc_std_{i}' for i in range(13)] +
              [f'delta_mean_{i}' for i in range(13)] + [f'delta_std_{i}' for i in range(13)] +
              [f'delta2_mean_{i}' for i in range(13)] + [f'delta2_std_{i}' for i in range(13)] +
              ['rmse_mean', 'rmse_std', 'spec_mean', 'spec_std', 'mel_mean', 'mel_std'])
    df = pd.DataFrame(features_list, columns=columns)
    df['emotion'] = labels
    scaler = StandardScaler()
    df[columns] = scaler.fit_transform(df[columns])
    df.to_csv(OUT_DIR_CSV, index=False)
    logging.info(f"DF guardado en {OUT_DIR_CSV}: {len(df)} filas")

else:
  print("La generacion del archivo CSV está desactivada")

NameError: name 'DATASET_PATH' is not defined